# 06 — Manuscript number audit

**Public reproducibility version.** This notebook preserves the final analytical logic used for the revised manuscript. Local absolute paths and development-only cells were cleaned for portability. The notebook has not been re-executed as part of repository packaging.


# 06 — Manuscript Number Audit (FINAL)

This notebook is a lightweight quality-control check for the manuscript-facing cohort numbers.

It follows the same final 17-keyword filtering logic used in the corrected main pipeline and reports the quantities used in the current manuscript:

- 19,303 retained post records
- 11,164 unique users/authors
- 13,065 records with at least one reply/repost/quote
- 6,238 records without these interactions
- 14,265 total reply/repost/quote interaction instances
- 681,664 total likes

This notebook does **not** rerun the network, semantic, community-detection, or bipartite analyses.


In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd
import re

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR

DATA_DIR = PROJECT_ROOT / "data" / "processed"
PARQUET_FILE = "matched_subset_exact_fuzzy_no_political_5_7_new.parquet"
RAW_PATH = DATA_DIR / PARQUET_FILE

AUDIT_DIR = PROJECT_ROOT / "outputs" / "manuscript_number_audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print("Source parquet:", RAW_PATH)
print("Audit output  :", AUDIT_DIR)

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Source parquet not found: {RAW_PATH}. Run Notebook 00 first or place the frozen handoff parquet in data/processed/."
    )


In [ ]:
# Load the same source parquet used for the corrected revision pipeline.
df = pd.read_parquet(RAW_PATH)
df["text"] = df["text"].astype(str)

print("Raw source rows:", f"{len(df):,}")


In [ ]:
# Final retained 17-keyword specification used for the revised analysis.

ORIGINAL_KEYWORDS_20 = [
    "Alzheimer", "dementia", "ADRD",
    "brain fog", "cognitive decline", "forgetfulness", "cognitive test", "memory loss",
    "apoe", "amyloid", "Aducanumab", "Donanemab", "Lecanemab",
    "Donepezil", "Memantine", "Risperidone", "foggy brain",
    "cognitive fog", "mild cognitive", "cognitive training"
]

REMOVED_NONSPECIFIC_KEYWORDS = [
    "brain fog",
    "foggy brain",
    "cognitive fog",
]

keywords = [
    kw for kw in ORIGINAL_KEYWORDS_20
    if kw.lower() not in {x.lower() for x in REMOVED_NONSPECIFIC_KEYWORDS}
]

exact_match_keywords = {"ADRD", "apoe"}
exact_match_lower = {k.lower() for k in exact_match_keywords}

def contains_AD_keyword(text):
    text = "" if pd.isna(text) else str(text)
    return any(
        re.search(
            rf"\b{re.escape(kw)}\b" if kw.lower() in exact_match_lower
            else rf"{re.escape(kw)}[\u2019's]*",
            text,
            flags=re.IGNORECASE
        )
        for kw in keywords
    )

mask = df["text"].apply(contains_AD_keyword)
final_cohort = df.loc[mask].copy()

# Same political-noise exclusion used in the corrected main pipeline.
final_cohort = final_cohort.loc[
    ~final_cohort["text"].str.contains(
        "mcconnell|reagan|ebony",
        case=False,
        na=False,
        regex=True
    )
].copy()

print("Final keyword count:", len(keywords))
print("Final retained post records:", f"{len(final_cohort):,}")


In [ ]:
# Manuscript-facing cohort quantities.

n_records = int(len(final_cohort))
n_users = int(final_cohort["user_id"].nunique())
total_likes = int(final_cohort["like_count"].fillna(0).sum())

interaction_cols = ["replied_author", "reposted_author", "quoted_author"]
has_interaction = final_cohort[interaction_cols].notna().any(axis=1)

n_interaction_records = int(has_interaction.sum())
n_noninteraction_records = int((~has_interaction).sum())

def detect_interaction_types(row):
    out = []
    if pd.notnull(row["replied_author"]):
        out.append("reply")
    if pd.notnull(row["reposted_author"]):
        out.append("repost")
    if pd.notnull(row["quoted_author"]):
        out.append("quote")
    return out

interaction_counts = Counter(
    interaction_type
    for types in final_cohort.apply(detect_interaction_types, axis=1)
    for interaction_type in types
)

n_reply = int(interaction_counts.get("reply", 0))
n_repost = int(interaction_counts.get("repost", 0))
n_quote = int(interaction_counts.get("quote", 0))
n_touchpoints = n_reply + n_repost + n_quote

print("=== MANUSCRIPT NUMBER AUDIT ===")
print(f"Retained post records : {n_records:,}")
print(f"Unique users/authors  : {n_users:,}")
print(f"Interaction records   : {n_interaction_records:,} ({n_interaction_records/n_records:.2%})")
print(f"Non-interaction       : {n_noninteraction_records:,} ({n_noninteraction_records/n_records:.2%})")
print(f"Reply instances       : {n_reply:,}")
print(f"Repost instances      : {n_repost:,}")
print(f"Quote instances       : {n_quote:,}")
print(f"Engagement touchpoints: {n_touchpoints:,}")
print(f"Total likes           : {total_likes:,}")


In [ ]:
# Consistency checks against the current frozen manuscript version.

EXPECTED = {
    "retained_post_records": 19303,
    "unique_users": 11164,
    "interaction_records": 13065,
    "noninteraction_records": 6238,
    "reply_instances": 6359,
    "repost_instances": 6590,
    "quote_instances": 1316,
    "engagement_touchpoints": 14265,
    "total_likes": 681664,
}

OBSERVED = {
    "retained_post_records": n_records,
    "unique_users": n_users,
    "interaction_records": n_interaction_records,
    "noninteraction_records": n_noninteraction_records,
    "reply_instances": n_reply,
    "repost_instances": n_repost,
    "quote_instances": n_quote,
    "engagement_touchpoints": n_touchpoints,
    "total_likes": total_likes,
}

checks = pd.DataFrame({
    "metric": EXPECTED.keys(),
    "expected": EXPECTED.values(),
    "observed": [OBSERVED[k] for k in EXPECTED],
})
checks["match"] = checks["expected"] == checks["observed"]

display(checks)

if not checks["match"].all():
    bad = checks.loc[~checks["match"]]
    raise RuntimeError(
        "One or more manuscript-facing numbers do not match the frozen values:\n"
        + bad.to_string(index=False)
    )

print("\nPASS: all manuscript-facing cohort numbers match the current frozen revision values.")


In [ ]:
# Save the audit table for future manuscript/rebuttal checking.

audit = pd.DataFrame([
    {"metric": "retained_post_records", "value": n_records},
    {"metric": "unique_users_authors", "value": n_users},
    {"metric": "interaction_records", "value": n_interaction_records},
    {"metric": "interaction_records_percent", "value": n_interaction_records / n_records},
    {"metric": "noninteraction_records", "value": n_noninteraction_records},
    {"metric": "noninteraction_records_percent", "value": n_noninteraction_records / n_records},
    {"metric": "reply_instances", "value": n_reply},
    {"metric": "repost_instances", "value": n_repost},
    {"metric": "quote_instances", "value": n_quote},
    {"metric": "engagement_touchpoints", "value": n_touchpoints},
    {"metric": "total_likes", "value": total_likes},
])

out_csv = AUDIT_DIR / "manuscript_number_audit_FINAL.csv"
audit.to_csv(out_csv, index=False)

display(audit)
print("\nSaved:", out_csv)


## Manuscript wording corresponding to this audit

Current cohort-description sentence:

> Our final ADRD data cohort consists of 19,303 posts from 11,164 users, totaling 14,265 engagement touchpoints. All original posts collectively received 681,664 likes.

The notebook is intended only to verify these descriptive quantities. It does not modify any downstream analysis.
